# Parameter Sensitivity Stability Pipeline Demo

Demonstrates the T009-T012 features:
- `compute_neighbor_smoothing()` — grid-aware smoothing + stability ratio
- `identify_stable_regions()` — BFS connected component detection
- `plot_parameter_sensitivity_with_stability()` — 1D raw vs smoothed + shaded regions
- `plot_2d_stability_heatmap()` — 2D heatmap + stability contour overlay
- `plot_3d_slices()` — 3D+ dropdown/slider 2D slices
- `generate_parameter_sensitivity_report()` — full orchestration pipeline

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd

from eda.parameter_analysis import (
    compute_neighbor_smoothing,
    identify_neighbors,
    identify_stable_regions,
    generate_parameter_sensitivity_report,
    StableRegion,
    ParameterSensitivityReport,
)
from metrics.plotting.parameter_plots import (
    plot_parameter_sensitivity_with_stability,
    plot_2d_stability_heatmap,
    plot_3d_slices,
)

print('All imports successful')

All imports successful


---
## 1D Example: RSI Lookback Sweep

Simulates a grid search over RSI lookback periods with a clear stable plateau around 12-18 and an isolated peak at 25.

In [2]:
# Synthetic 1D grid: stable plateau at 12-18, isolated peak at 25
lookbacks = list(range(5, 31))
np.random.seed(42)

sortino_1d = []
for lb in lookbacks:
    if 12 <= lb <= 18:
        val = 1.0 + 0.15 * np.sin((lb - 12) * 0.5) + np.random.normal(0, 0.03)
    elif lb == 25:
        val = 1.3  # isolated overfit peak
    else:
        val = 0.4 + np.random.normal(0, 0.05)
    sortino_1d.append(val)

df_1d = pd.DataFrame({
    'param1_value': lookbacks,
    'sortino': sortino_1d,
})

print(f'Grid: {len(df_1d)} parameter values')
df_1d.head(10)

Grid: 26 parameter values


,param1_value,sortino
0,5,0.424836
1,6,0.393087
2,7,0.432384
3,8,0.476151
4,9,0.388292
5,10,0.388293
6,11,0.478961
7,12,1.023023
8,13,1.057830
9,14,1.142497


In [3]:
# Step 1: Neighbor smoothing
smoothed_1d = compute_neighbor_smoothing(df_1d, ['lookback'], 'sortino')
smoothed_1d[['param1_value', 'sortino', 'smoothed_sortino', 'stability_ratio', 'n_neighbors']]

,param1_value,sortino,smoothed_sortino,stability_ratio,n_neighbors
0,5,0.424836,0.408961,0.962634,1
1,6,0.393087,0.416769,1.060247,2
2,7,0.432384,0.433874,1.003446,2
3,8,0.476151,0.432276,0.907854,2
4,9,0.388292,0.417579,1.075424,2
5,10,0.388293,0.418515,1.077834,2
6,11,0.478961,0.630092,1.315541,2
7,12,1.023023,0.853271,0.834068,2
8,13,1.057830,1.074450,1.015712,2
9,14,1.142497,1.112016,0.973321,2


In [4]:
# Step 2: Identify stable regions
regions_1d = identify_stable_regions(smoothed_1d, 'sortino', stability_threshold=0.8)

for i, r in enumerate(regions_1d):
    vals = sorted([c[0] for c in r.param_combinations])
    print(f'Region {i}: params={vals}, mean_obj={r.mean_objective:.3f}, '
          f'stability={r.mean_stability_ratio:.3f}, boundary={r.is_boundary_region}')

Region 0: params=[np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)], mean_obj=0.649, stability=1.086, boundary=True
Region 1: params=[np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30)], mean_obj=0.447, stability=1.119, boundary=True


In [5]:
# Step 3: 1D stability plot
fig_1d = plot_parameter_sensitivity_with_stability(
    smoothed_1d, 'RSI Lookback', 'sortino',
    stable_regions=regions_1d,
    show_plot=True,
)

**What to look for:**
- Green shaded band over the 12-18 plateau (genuine stable region)
- The isolated peak at 25 is NOT shaded (overfit — neighbors are poor)
- Dashed smoothed line converges with raw in stable zones, drops away from peaks

---
## 2D Example: EWMAC Fast x Slow Grid

Simulates a 2D parameter grid with a stable plateau in the center-right region.

In [6]:
# Synthetic 2D grid: stable plateau where fast=16-32 and slow=64-128
fast_vals = [4, 8, 16, 32, 64]
slow_vals = [16, 32, 64, 128, 256]
np.random.seed(7)

rows_2d = []
for f in fast_vals:
    for s in slow_vals:
        if 16 <= f <= 32 and 64 <= s <= 128:
            val = 1.2 + np.random.normal(0, 0.05)  # stable plateau
        elif f == 4 and s == 256:
            val = 1.5  # isolated corner peak
        else:
            val = 0.4 + np.random.normal(0, 0.08)
        rows_2d.append({'param1_value': f, 'param2_value': s, 'sortino': val})

df_2d = pd.DataFrame(rows_2d)
print(f'Grid: {len(df_2d)} combinations ({len(fast_vals)}x{len(slow_vals)})')
df_2d.pivot_table(index='param1_value', columns='param2_value', values='sortino').round(2)

Grid: 25 combinations (5x5)


param2_value,16,32,64,128,256
param1_value,,,,,
4,0.54,0.36,0.40,0.43,1.50
8,0.34,0.40,0.40,0.26,0.48
16,0.45,0.35,1.19,1.23,0.38
32,0.38,0.28,1.23,1.21,0.42
64,0.28,0.53,0.41,0.37,0.56


In [7]:
# Smooth + find regions
smoothed_2d = compute_neighbor_smoothing(df_2d, ['fast', 'slow'], 'sortino')
regions_2d = identify_stable_regions(smoothed_2d, 'sortino', stability_threshold=0.8)

for i, r in enumerate(regions_2d):
    print(f'Region {i}: n={r.n_combinations}, ranges={r.param_ranges}, '
          f'mean_obj={r.mean_objective:.3f}, boundary={r.is_boundary_region}')

Region 0: n=18, ranges={'param1': (np.int64(4), np.int64(64)), 'param2': (np.int64(16), np.int64(256))}, mean_obj=0.512, boundary=True


In [8]:
# 2D stability heatmap
fig_2d = plot_2d_stability_heatmap(
    smoothed_2d, 'Fast Period', 'Slow Period', 'sortino',
    stable_regions=regions_2d,
    show_plot=True,
)

**What to look for:**
- Viridis heatmap showing smoothed Sortino values
- RdYlGn contour overlay showing stability ratio (green = stable)
- Green square markers around the stable plateau (fast=16-32, slow=64-128)
- The isolated corner peak at (4, 256) should NOT be marked as stable

---
## 3D Example: Three-Parameter Grid

Simulates a 3D grid (lookback x threshold x smoothing) with interactive slicing.

In [9]:
# Synthetic 3D grid
lb_vals = [5, 10, 15, 20]
th_vals = [0.3, 0.5, 0.7, 0.9]
sm_vals = [3, 5, 7, 9]
np.random.seed(99)

rows_3d = []
for lb in lb_vals:
    for th in th_vals:
        for sm in sm_vals:
            # Peak performance around lb=10-15, th=0.5-0.7, sm=5-7
            dist = abs(lb - 12.5)/10 + abs(th - 0.6)/0.5 + abs(sm - 6)/5
            val = max(0.1, 1.3 - dist + np.random.normal(0, 0.05))
            rows_3d.append({
                'param1_value': lb,
                'param2_value': th,
                'param3_value': sm,
                'sortino': val,
            })

df_3d = pd.DataFrame(rows_3d)
print(f'Grid: {len(df_3d)} combinations ({len(lb_vals)}x{len(th_vals)}x{len(sm_vals)})')

Grid: 64 combinations (4x4x4)


In [10]:
# Smooth + find regions
smoothed_3d = compute_neighbor_smoothing(df_3d, ['lookback', 'threshold', 'smoothing'], 'sortino')
regions_3d = identify_stable_regions(smoothed_3d, 'sortino', stability_threshold=0.8)

print(f'Found {len(regions_3d)} stable region(s)')
for i, r in enumerate(regions_3d):
    print(f'  Region {i}: n={r.n_combinations}, mean_obj={r.mean_objective:.3f}, '
          f'stability={r.mean_stability_ratio:.3f}')
    for name, (lo, hi) in r.param_ranges.items():
        print(f'    {name}: [{lo}, {hi}]')

Found 1 stable region(s)
  Region 0: n=57, mean_obj=0.194, stability=1.284
    param1: [5, 20]
    param2: [0.3, 0.9]
    param3: [3, 9]


In [11]:
# 3D slicing plot: fix one param, see 2D heatmap of the others
fig_3d = plot_3d_slices(
    smoothed_3d,
    ['Lookback', 'Threshold', 'Smoothing'],
    'sortino',
    show_plot=True,
)

**What to look for:**
- Dropdown at top-left: "Fix Lookback", "Fix Threshold", "Fix Smoothing"
- Slider at bottom: step through values of the fixed parameter
- Heatmap shows the 2D slice of the other two params
- Try fixing Lookback and sliding — you should see the warm zone shift

---
## Full Report Pipeline (1D)

One-call orchestration: smoothing → region detection → plotting → recommendations.

In [12]:
report_1d = generate_parameter_sensitivity_report(
    df_1d, ['lookback'], 'sortino',
    stability_threshold=0.8,
    top_k=3,
)

print(f'Metric: {report_1d.metric_name}')
print(f'Grid size: {report_1d.n_parameter_combinations}')
print(f'Stable regions: {report_1d.n_stable_regions}')
print(f'Mean stability ratio: {report_1d.mean_stability_ratio:.3f}')
print(f'% stable combinations: {report_1d.pct_stable_combinations:.1%}')
print(f'Top-3 recommendations: {report_1d.top_k_combinations}')
print(f'Timestamp: {report_1d.timestamp}')

Metric: sortino
Grid size: 26
Stable regions: 2
Mean stability ratio: 1.071
% stable combinations: 96.2%
Top-3 recommendations: [(np.int64(15),), (np.int64(16),), (np.int64(14),)]
Timestamp: 2026-02-16T07:29:09.966866+00:00


In [13]:
# The report auto-generates the appropriate plot
report_1d.plot_1d.show()

---
## Full Report Pipeline (2D)

In [14]:
report_2d = generate_parameter_sensitivity_report(
    df_2d, ['fast', 'slow'], 'sortino',
    stability_threshold=0.8,
    top_k=5,
)

print(f'Stable regions: {report_2d.n_stable_regions}')
print(f'% stable: {report_2d.pct_stable_combinations:.1%}')
print(f'Top-5 recommendations: {report_2d.top_k_combinations}')
print(f'plot_1d={report_2d.plot_1d}, plot_2d={type(report_2d.plot_2d).__name__}, plot_3d={report_2d.plot_3d}')

Stable regions: 1
% stable: 72.0%
Top-5 recommendations: [(np.int64(8), np.int64(256)), (np.int64(4), np.int64(128)), (np.int64(32), np.int64(256)), (np.int64(64), np.int64(128)), (np.int64(64), np.int64(64))]
plot_1d=None, plot_2d=Figure, plot_3d=None


In [15]:
report_2d.plot_2d.show()

---
## Full Report Pipeline (3D)

In [16]:
report_3d = generate_parameter_sensitivity_report(
    df_3d, ['lookback', 'threshold', 'smoothing'], 'sortino',
    stability_threshold=0.8,
    top_k=5,
)

print(f'Stable regions: {report_3d.n_stable_regions}')
print(f'% stable: {report_3d.pct_stable_combinations:.1%}')
print(f'Top-5 recommendations: {report_3d.top_k_combinations}')
print(f'plot_1d={report_3d.plot_1d}, plot_2d={report_3d.plot_2d}, plot_3d={type(report_3d.plot_3d).__name__}')

Stable regions: 1
% stable: 89.1%
Top-5 recommendations: [(np.float64(15.0), np.float64(0.7), np.float64(5.0)), (np.float64(15.0), np.float64(0.5), np.float64(7.0)), (np.float64(15.0), np.float64(0.7), np.float64(9.0)), (np.float64(15.0), np.float64(0.9), np.float64(7.0)), (np.float64(10.0), np.float64(0.9), np.float64(5.0))]
plot_1d=None, plot_2d=None, plot_3d=Figure


In [17]:
report_3d.plot_3d.show()

---
## Verification Checklist

After running all cells, confirm:

- [ ] **1D plot**: Green shaded band over lookback 12-18, isolated peak at 25 NOT shaded
- [ ] **1D plot**: Dashed smoothed line tracks raw in stable zone, diverges at edges
- [ ] **2D heatmap**: Warm Viridis region at fast=16-32, slow=64-128
- [ ] **2D heatmap**: Green stability contour covers the plateau, not the corner peak
- [ ] **2D heatmap**: Square markers highlight the stable region points
- [ ] **3D slices**: Dropdown works (Fix Lookback / Threshold / Smoothing)
- [ ] **3D slices**: Slider steps through fixed param values, heatmap updates
- [ ] **Reports**: `top_k_combinations` are within stable regions
- [ ] **Reports**: Only the matching plot field is non-None (1D→plot_1d, 2D→plot_2d, 3D→plot_3d)